# SenseVoice 数据量消融实验

验证不同数据量（25%/50%/75%）和两种数据增强策略对微调效果的影响。

In [ ]:
def _resample_torch(wav: torch.Tensor, orig_sr: int, new_sr: int) -> torch.Tensor:
    """用 scipy.signal.resample 做高质量重采样"""
    if orig_sr == new_sr:
        return wav
    from scipy import signal
    num_samples = int(wav.shape[1] * new_sr / orig_sr)
    resampled = signal.resample(wav.numpy(), num_samples, axis=-1)
    return torch.from_numpy(resampled)

In [ ]:
# --- 生成 5 个实验组的数据 ---
print("=" * 60)
print("开始生成实验数据...")
print("=" * 60)

# 实验组 1-3: 子采样
pct_configs = [
    (0.25, "/mnt/data/prepared_data/sv_ablation/train_25pct.jsonl"),
    (0.50, "/mnt/data/prepared_data/sv_ablation/train_50pct.jsonl"),
    (0.75, "/mnt/data/prepared_data/sv_ablation/train_75pct.jsonl"),
]

for ratio, out_path in pct_configs:
    subsample(raw_train, ratio, out_path)

# 实验组 4: 合成噪音增强 (A) → 200%
train_noise_a_path = generate_noise_a_dataset(
    raw_train,
    "/mnt/data/prepared_data/sv_ablation",
    "train_noise_a.jsonl"
)

# 实验组 5: 真实噪音增强 (B) → 200%
train_noise_b_path = generate_noise_b_dataset(
    raw_train,
    "/mnt/data/prepared_data/sv_ablation",
    "train_noise_b.jsonl"
)

print("\n数据生成完成!")
print(f"  25%:  {pct_configs[0][1]}")
print(f"  50%:  {pct_configs[1][1]}")
print(f"  75%:  {pct_configs[2][1]}")
print(f"  噪声A: {train_noise_a_path}")
print(f"  噪声B: {train_noise_b_path if train_noise_b_path else '(跳过: DEMAND噪音未找到)'}")

In [ ]:
import subprocess
import sys

def train_sensevoice_lora(train_jsonl, output_dir, exp_name):
    """训练 SenseVoice-Small LoRA"""
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        "torchrun", "--nproc_per_node=1", "-m", "funasr.bin.train_ds",
        "++model=iic/SenseVoiceSmall",
        f"++train_data_set_list={train_jsonl}",
        f"++valid_data_set_list={VAL_JSONL}",
        # data_split_num=1: 使用完整的 val.jsonl 作为验证集，不做额外分割
        "++dataset_conf.data_split_num=1",
        "++dataset_conf.batch_sampler=BatchSampler",
        "++dataset_conf.batch_size=6000",
        "++dataset_conf.sort_size=1024",
        "++dataset_conf.batch_type=token",
        "++dataset_conf.num_workers=4",
        "++train_conf.max_epoch=10",
        "++train_conf.log_interval=50",
        "++train_conf.validate_interval=2000",
        "++train_conf.save_checkpoint_interval=2000",
        "++train_conf.keep_nbest_models=5",
        "++train_conf.avg_nbest_model=3",
        "++train_conf.use_bf16=true",
        "++train_conf.grad_clip=5",
        "++lora_only=true",
        "++lora_bias=none",
        "++lora_rank=8",
        "++lora_alpha=16",
        "++lora_dropout=0.1",
        "++optim=adamw",
        "++optim_conf.lr=2e-4",
        "++optim_conf.weight_decay=0.0",
        f"++output_dir={output_dir}",
    ]

    print(f"\n{'='*60}")
    print(f"训练: {exp_name}")
    print(f"数据: {train_jsonl}")
    print(f"输出: {output_dir}")
    print(f"{'='*60}")

    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()

    returncode = process.wait()
    if returncode == 0:
        print(f"✓ {exp_name} 训练完成!")
    else:
        print(f"✗ {exp_name} 训练失败! 返回码: {returncode}")
    return returncode == 0

# --- 处理噪声数据路径（可能被前面cell跳过） ---
try:
    train_noise_a_path
except NameError:
    train_noise_a_path = None
try:
    train_noise_b_path
except NameError:
    train_noise_b_path = None

# --- 5 个实验组配置 ---
experiments = [
    ("25%",   "/mnt/data/prepared_data/sv_ablation/train_25pct.jsonl",   "/mnt/output/sv_25pct",   "sv_25pct"),
    ("50%",   "/mnt/data/prepared_data/sv_ablation/train_50pct.jsonl",   "/mnt/output/sv_50pct",   "sv_50pct"),
    ("75%",   "/mnt/data/prepared_data/sv_ablation/train_75pct.jsonl",   "/mnt/output/sv_75pct",   "sv_75pct"),
]
if train_noise_a_path:
    experiments.append(("噪声A", train_noise_a_path, "/mnt/output/sv_noise_a", "sv_noise_a"))
if train_noise_b_path:
    experiments.append(("噪声B", train_noise_b_path, "/mnt/output/sv_noise_b", "sv_noise_b"))

# --- 逐一训练（可跳过已完成的） ---
training_results = {}
for name, train_jsonl, output_dir, ckpt_name in experiments:
    ckpt_path = f"{output_dir}/model.pt.best"
    if os.path.exists(ckpt_path):
        print(f"\n[跳过] {name} 已完成，checkpoint 存在")
        training_results[name] = ckpt_path
        continue
    success = train_sensevoice_lora(train_jsonl, output_dir, f"{name} ({ckpt_name})")
    training_results[name] = ckpt_path if success else None

print("\n训练汇总:")
for name, ckpt in training_results.items():
    status = "✓" if ckpt and os.path.exists(ckpt) else "✗"
    print(f"  {status} {name}: {ckpt}")

In [ ]:
from funasr import AutoModel
import re
import time
import gc

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def clean_sensevoice_text(text):
    if not text:
        return ""
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 加载验证集
samples = []
with open(VAL_JSONL) as f:
    for line in f:
        samples.append(json.loads(line))
valid_samples = [s for s in samples if os.path.exists(s['source'])]
print(f"验证集: {len(samples)} 条, 有效: {len(valid_samples)} 条")

# 加载基座模型（用于 baseline 对比）
print("加载 SenseVoice 基座模型...")
sv_base = AutoModel(model="iic/SenseVoiceSmall", disable_update=True)

def eval_model(model, samples, label):
    """评估单个模型，返回 CER"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio, expected = s['source'], s['target']
            if not os.path.exists(audio):
                continue
            try:
                res = model.generate(input=audio, language="auto", use_itn=True)
                pred = clean_sensevoice_text(res[0]['text']) if res else ""
            except:
                pred = ""
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            results.append({"id": i, "expected": expected, "predicted": pred, "cer": cer})
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    print(f"  {label}: CER={cer:.2%}, 精确={exact}/{len(results)}, 耗时={elapsed:.1f}s")
    return {"name": label, "cer": cer, "exact": exact, "total": len(results), "time": elapsed, "results": results}

# 基座模型评估
print("\n评估基座模型...")
sv_base_result = eval_model(sv_base, valid_samples, "SV-base")

In [ ]:
# 评估所有微调模型
all_results = [sv_base_result]

for name, ckpt_path in training_results.items():
    if ckpt_path is None or not os.path.exists(ckpt_path):
        print(f"\n跳过 {name}: checkpoint 不存在")
        continue

    print(f"\n加载并评估 {name}...")
    try:
        # 加载微调模型
        model = AutoModel(model="iic/SenseVoiceSmall", lora_only=True, disable_update=True)
        ckpt = torch.load(ckpt_path, map_location="cpu")
        model.model.load_state_dict(ckpt["state_dict"], strict=False)
        del ckpt

        # 评估
        result = eval_model(model, valid_samples, f"SV-ft-{name}")
        all_results.append(result)
    except Exception as e:
        print(f"  警告: {name} 评估失败 - {e}")
        continue
    finally:
        # 释放显存
        if 'model' in dir():
            del model
        free_gpu()

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- 汇总表格 ---
print("\n" + "=" * 70)
print("数据量消融实验 CER 汇总")
print("=" * 70)
print(f"{'实验组':<16} {'CER':>10} {'精确匹配':>10} {'样本数':>8} {'相对基座提升':>12}")
print("-" * 70)

base_cer = sv_base_result['cer']
for r in all_results:
    improve = (base_cer - r['cer']) / base_cer * 100 if base_cer > 0 else 0
    print(f"{r['name']:<16} {r['cer']:>9.2%} {r['exact']:>9}/{r['total']:<8} {improve:>+10.1f}%")
print("=" * 70)

# --- 可视化 ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('SenseVoice 数据量消融实验', fontsize=16, fontweight='bold')

names = [r['name'] for r in all_results]
cers = [r['cer'] * 100 for r in all_results]
colors = ['#4ECDC4'] + ['#FF8E53', '#FF6B6B', '#45B7D1', '#96CEB4', '#FFEAA7'][:len(all_results)-1]

# CER 柱状图
bars = axes[0].bar(names, cers, color=colors[:len(names)])
axes[0].set_title('CER 对比')
axes[0].set_ylabel('CER (%)')
axes[0].set_ylim(0, max(cers) * 1.3)
for bar, cer in zip(bars, cers):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3, f'{cer:.1f}%', ha='center', va='bottom')
axes[0].tick_params(axis='x', rotation=15)

# 相对基座提升
improvements = [(base_cer - r['cer']) / base_cer * 100 if base_cer > 0 else 0 for r in all_results[1:]]
exp_names = [r['name'] for r in all_results[1:]]
bar_colors = ['#52c41a' if x > 0 else '#ff4d4f' for x in improvements]
bars2 = axes[1].bar(exp_names, improvements, color=bar_colors)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('相对基座提升 (%)')
axes[1].set_ylabel('提升 (%)')
for bar, imp in zip(bars2, improvements):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f'{imp:+.1f}%', ha='center', va='bottom')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
chart_path = "/mnt/output/sv_data_ablation.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n图表已保存: {chart_path}")

# --- 保存结果 JSON ---
result_path = "/mnt/output/sv_data_ablation.json"
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump({
        "base_cer": base_cer,
        "results": [
            {
                "name": r['name'],
                "cer": round(r['cer'], 4),
                "exact_match": r['exact'],
                "total": r['total'],
                "improvement_pct": round((base_cer - r['cer']) / base_cer * 100, 2) if base_cer > 0 else 0,
            } for r in all_results
        ]
    }, f, ensure_ascii=False, indent=2)
print(f"结果已保存: {result_path}")